<a href="https://colab.research.google.com/github/UmitKayaardi/car_accident_analysis/blob/main/car_accident_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Bu kod edited_videos içindeki videoları ayırmaya yaran kod.

import os
import shutil
import random
from google.colab import drive
drive.mount('/content/drive')

def split_videos(source_dir, target_dir, train_ratio=0.7, validation_ratio=0.15, test_ratio=0.15):
    """
    Videoları küçük, orta, büyük seviyelerine göre train, validation, test klasörlerine dağıtır.
    source_dir: Kaynak klasör (edited_videos)
    target_dir: Hedef ana klasör (trafik_dataset)
    train_ratio, validation_ratio, test_ratio: Dağıtım oranları
    """
    # Küçük, Orta, Büyük klasörlerini işleme
    for level in os.listdir(source_dir):  # Örn: kucuk, orta, buyuk
        level_path = os.path.join(source_dir, level)
        if not os.path.isdir(level_path):
            continue

        # Videoların tam listesi
        videos = os.listdir(level_path)
        random.shuffle(videos)  # Videoları karıştırma

        # Dağıtım sayıları
        total_videos = len(videos)
        train_count = int(total_videos * train_ratio)
        validation_count = int(total_videos * validation_ratio)
        test_count = total_videos - train_count - validation_count

        # Train, Validation, Test klasörlerine dağıtım
        split_dict = {
            "train": videos[:train_count],
            "validation": videos[train_count:train_count + validation_count],
            "test": videos[train_count + validation_count:]
        }

        for split, video_list in split_dict.items():
            split_dir = os.path.join(target_dir, split, level)
            os.makedirs(split_dir, exist_ok=True)
            for video in video_list:
                src_path = os.path.join(level_path, video)
                dst_path = os.path.join(split_dir, video)
                shutil.copy(src_path, dst_path)
                print(f"Kopyalandı: {src_path} -> {dst_path}")

# Klasör yolları
source_dir = "/content/drive/My Drive/trafik_dataset/edited_videos"  # Düzenlenmiş videoların olduğu yer
target_dir = "/content/drive/My Drive/trafik_dataset"  # Ana dataset klasörü

# Dağıtımı gerçekleştir
split_videos(source_dir, target_dir)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Kopyalandı: /content/drive/My Drive/trafik_dataset/edited_videos/buyuk/2019_08_22.mp4 -> /content/drive/My Drive/trafik_dataset/train/buyuk/2019_08_22.mp4
Kopyalandı: /content/drive/My Drive/trafik_dataset/edited_videos/buyuk/2019_08_3.mp4 -> /content/drive/My Drive/trafik_dataset/train/buyuk/2019_08_3.mp4
Kopyalandı: /content/drive/My Drive/trafik_dataset/edited_videos/buyuk/26.mp4 -> /content/drive/My Drive/trafik_dataset/train/buyuk/26.mp4
Kopyalandı: /content/drive/My Drive/trafik_dataset/edited_videos/buyuk/19.mp4 -> /content/drive/My Drive/trafik_dataset/train/buyuk/19.mp4
Kopyalandı: /content/drive/My Drive/trafik_dataset/edited_videos/buyuk/2019_07_30.mp4 -> /content/drive/My Drive/trafik_dataset/train/buyuk/2019_07_30.mp4
Kopyalandı: /content/drive/My Drive/trafik_dataset/edited_videos/buyuk/43.mp4 -> /content/drive/My Drive/trafik_dataset/train/buyu

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv3D, MaxPooling3D, Flatten, Dense, Dropout, LSTM, TimeDistributed, RepeatVector, Concatenate, Lambda, Permute, Reshape
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import backend as K

# 1. Optical Flow Hesaplama
def extract_optical_flow(video_path, frame_count=30):
    """
    Bir videodan optical flow özelliklerini çıkarır ve numpy dizisi olarak döndürür.
    """
    cap = cv2.VideoCapture(video_path)
    ret, prev_frame = cap.read()

    if not ret:
        return np.zeros((frame_count, 128, 128, 2))  # Eğer video okunamazsa boş bir dizi döndür

    # İlk frame'i gri tonlamaya çevir
    prev_frame = cv2.cvtColor(cv2.resize(prev_frame, (128, 128)), cv2.COLOR_BGR2GRAY)
    flows = []

    for _ in range(frame_count - 1):  # Optical flow için bir frame daha az alıyoruz
        ret, frame = cap.read()
        if not ret:
            break

        # Mevcut frame'i gri tonlamaya çevir
        current_frame = cv2.cvtColor(cv2.resize(frame, (128, 128)), cv2.COLOR_BGR2GRAY)

        # Optical flow hesapla
        flow = cv2.calcOpticalFlowFarneback(prev_frame, current_frame, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        flows.append(flow)

        # Sonraki iterasyon için güncelleme
        prev_frame = current_frame

    cap.release()

    # Eğer yeterince frame yoksa, diziyi tamamlamak için sıfır ekleyelim
    while len(flows) < frame_count - 1:
        flows.append(np.zeros((128, 128, 2)))

    flows = np.array(flows)
    return flows

# 2. Veri Yükleme
def load_data_with_optical_flow(data_dir, frame_count=30):
    """
    Optical Flow kullanarak veriyi yükler.
    """
    X, y = [], []
    labels = {'kucuk': 0, 'orta': 1, 'buyuk': 2}

    for label, idx in labels.items():
        path = os.path.join(data_dir, label)
        for file in os.listdir(path):
            video_path = os.path.join(path, file)
            flow = extract_optical_flow(video_path, frame_count)
            X.append(flow)
            y.append(idx)

    X = np.array(X)
    y = to_categorical(y, num_classes=3)
    return X, y

# 3. Model Tanımlama
def attention_3d_block(inputs):
    """
    Temporal attention mekanizması (boyut uyumsuzlukları giderildi).
    """
    time_steps = K.int_shape(inputs)[1]  # Zaman adımlarını alın
    input_dim = K.int_shape(inputs)[2]  # Giriş boyutunu alın

    # Dikkat ağırlıkları: (batch_size, time_steps)
    a = Dense(1, activation='softmax')(inputs)
    a = Lambda(lambda x: K.squeeze(x, axis=-1))(a)  # (batch_size, time_steps)

    # Boyutları yer değiştirme ve dikkat ağırlıklarını girişlere uygulama
    a_probs = Lambda(lambda x: K.expand_dims(x, axis=-1))(a)  # (batch_size, time_steps, 1)
    output_attention_mul = Lambda(lambda x: x[0] * x[1])([inputs, a_probs])  # Çarpım (batch_size, time_steps, input_dim)
    output_attention_sum = Lambda(lambda x: K.sum(x, axis=1))(output_attention_mul)  # Zaman üzerinde topla

    return output_attention_sum

def create_deeper_optical_flow_model(input_shape):
    """
    Daha derin Conv3D ve MaxPooling3D yapısı ile temporal attention kullanılarak oluşturulan model.
    """
    inputs = Input(shape=input_shape)

    # Conv3D + MaxPooling3D Katmanları (4 katman)
    conv3d_1 = Conv3D(32, (3, 3, 3), activation='relu')(inputs)
    pool3d_1 = MaxPooling3D((2, 2, 2))(conv3d_1)

    conv3d_2 = Conv3D(64, (3, 3, 3), activation='relu')(pool3d_1)
    pool3d_2 = MaxPooling3D((2, 2, 2))(conv3d_2)

    conv3d_3 = Conv3D(128, (3, 3, 3), activation='relu')(pool3d_2)
    pool3d_3 = MaxPooling3D((2, 2, 2))(conv3d_3)

    conv3d_4 = Conv3D(256, (3, 3, 3), activation='relu')(pool3d_3)
    pool3d_4 = MaxPooling3D((2, 2, 2))(conv3d_4)

    # Flatten ve LSTM
    flattened = TimeDistributed(Flatten())(pool3d_4)
    lstm_layer = LSTM(128, return_sequences=True)(flattened)

    # Attention Katmanı
    attention_output = attention_3d_block(lstm_layer)

    # Dense Katmanları
    dense_1 = Dense(256, activation='relu')(attention_output)
    dropout_1 = Dropout(0.5)(dense_1)
    dense_2 = Dense(128, activation='relu')(dropout_1)
    dropout_2 = Dropout(0.5)(dense_2)

    # Çıkış Katmanı
    output = Dense(3, activation='softmax')(dropout_2)  # 3 sınıf: Küçük, Orta, Büyük

    # Modelin Derlenmesi
    model = Model(inputs=inputs, outputs=output)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    return model

# 4. Eğitim ve Test
# Veri yolları
train_dir = '/content/drive/My Drive/trafik_dataset/train'
validation_dir = '/content/drive/My Drive/trafik_dataset/validation'
test_dir = '/content/drive/My Drive/trafik_dataset/test'

# Veriyi yükleme
print("Train verileri yükleniyor...")
X_train, y_train = load_data_with_optical_flow(train_dir)
print("Validation verileri yükleniyor...")
X_val, y_val = load_data_with_optical_flow(validation_dir)
print("Test verileri yükleniyor...")
X_test, y_test = load_data_with_optical_flow(test_dir)

# Modeli oluşturma
input_shape = (29, 128, 128, 2)  # Optical flow için 29 frame (n-1) ve 2 kanal (x ve y)
model = create_optical_flow_model(input_shape)

# Model eğitme
print("Model eğitiliyor...")
model.fit(X_train, y_train, epochs=25, validation_data=(X_val, y_val), batch_size=8)

# Modeli test etme
print("Model test ediliyor...")
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Doğruluğu: {test_accuracy * 100:.2f}%")

# 5. Yeni Video Sınıflandırma
def classify_video_with_optical_flow(video_path, model):
    """
    Yeni bir videoyu optical flow ile sınıflandırır.
    """
    flow = extract_optical_flow(video_path)
    flow = np.expand_dims(flow, axis=0)  # Modelin beklediği boyut
    prediction = model.predict(flow)
    class_labels = ['Küçük', 'Orta', 'Büyük']
    predicted_class = class_labels[np.argmax(prediction)]
    return predicted_class

# Yeni video sınıflandırma
new_video_path = '/content/drive/My Drive/trafik_dataset/new_video.mp4'
print("\n")
print("Yeni video sınıflandırılıyor...")
print(f"Yeni video sınıflandırması: {classify_video_with_optical_flow(new_video_path, model)}")


Train verileri yükleniyor...
Validation verileri yükleniyor...
Test verileri yükleniyor...
Model eğitiliyor...
Epoch 1/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 154s 19s/step - accuracy: 0.3817 - loss: 2.6383 - val_accuracy: 0.5833 - val_loss: 1.0525
Epoch 2/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 198s 18s/step - accuracy: 0.6321 - loss: 1.0281 - val_accuracy: 0.5000 - val_loss: 1.1492
Epoch 3/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 204s 19s/step - accuracy: 0.6574 - loss: 0.9021 - val_accuracy: 0.4167 - val_loss: 1.2099
Epoch 4/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 201s 19s/step - accuracy: 0.4568 - loss: 1.0902 - val_accuracy: 0.5000 - val_loss: 1.1614
Epoch 5/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 201s 18s/step - accuracy: 0.6213 - loss: 0.7991 - val_accuracy: 0.5000 - val_loss: 1.2404
Epoch 6/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 205s 19s/step - accuracy: 0.6697 - loss: 0.7248 - val_accuracy: 0.4167 - val_loss: 1.5132
Epoch 7/25
8/8 ━━━━━━━━━━━━━━━━━━━━ 149s 19s/step - accuracy: 0.8247 - loss: 0.5253 - val_accuracy: 0.5000 - val_loss: 1.9278
Epoch 8

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Yeni video sınıflandırması: Orta
